# Inferring Solutions of Differential Equations Using Noisy Multi-Fidelity Data

**Paper:** Raissi, M., Perdikaris, P., Karniadakis, G.E. (2017). *Inferring solutions of differential equations using noisy multi-fidelity data.* Journal of Computational Physics, 335, 736-746.

**Carpeta origen:** `PINNs/4. Otros/Inferring-solutions-of-differential-equations-usin_2017_Journal-of-Computati.pdf`

## Aclaracion importante: este paper NO usa PINNs (redes neuronales)

A diferencia de todos los demas cuadernos de esta coleccion, este paper de 2017 de Raissi, Perdikaris y Karniadakis &mdash; el mismo equipo que dos anos despues publicaria el paper fundacional de PINNs (Raissi et al. 2019, tambien en esta coleccion) &mdash; **no usa redes neuronales en absoluto**. En su lugar, propone **regresion con Procesos Gaussianos (GP)** cuyo *kernel* de covarianza se construye a partir del operador diferencial/integral lineal de la EDO/EDP (Eq. 1-4):

$$u(x)\sim\mathcal{GP}(0,g(x,x';\theta)), \qquad f(x)=\mathcal{L}_x[u](x)\sim\mathcal{GP}(0,k(x,x';\theta)),\qquad k(x,x';\theta):=\mathcal{L}_x\mathcal{L}_{x'}g(x,x';\theta)$$

Como $\mathcal{L}_x$ es lineal, aplicarlo a un GP produce **otro GP** con un kernel derivado analiticamente del kernel original. Esto permite condicionar la distribucion posterior de $u(x)$ usando **solo observaciones ruidosas y escasas de $f(x)$** (el termino forzante/dato de la EDO), sin discretizar el dominio ni requerir datos en la frontera &mdash; es un metodo Bayesiano, no una red neuronal entrenada por descenso de gradiente. El paper ademas incorpora **datos multi-fidelidad** (Eq. 1-13): observaciones baratas pero ruidosas de baja fidelidad $f_1(x)$ combinadas con pocas observaciones caras de alta fidelidad $f_2(x)$, mediante una estructura autoregresiva $u(x)=\rho u_1(x)+\delta_2(x)$.

Se incluye en esta carpeta "4. Otros" por su estrecha relacion historica y metodologica con las PINNs (es el trabajo previo directo, del mismo grupo, que dio lugar al framework de PINNs). Este cuaderno reproduce fielmente el **ejemplo pedagogico del paper (Seccion 4.1, Fig. 1)**, simplificado a la version de una sola fidelidad y un operador puramente diferencial (se omite el termino integral de la Eq. 17 del paper por brevedad, documentado explicitamente): $u'(x)=f(x)$, recuperando $u(x)=\sin(2\pi x)$ a partir de observaciones ruidosas de $f(x)=2\pi\cos(2\pi x)$ y un punto ancla en $u$, con cuantificacion de incertidumbre Bayesiana.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio en las paginas revisadas, ni fue posible confirmar uno especifico para este trabajo de 2017 en esta sesion. Como referencia del framework PINN posterior de los mismos autores (que sucede a este trabajo):

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
# Nota: este paper no usa redes neuronales (ver celda anterior), por lo que no requiere torch.
%pip install -q numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(0)

## 1. Ejemplo simplificado: $u'(x)=f(x)$, $u(x)=\sin(2\pi x)$, $f(x)=2\pi\cos(2\pi x)$

In [ ]:
def exact_u(x):
    return np.sin(2 * np.pi * x)

def exact_f(x):
    return 2 * np.pi * np.cos(2 * np.pi * x)

n_f = 15  # observaciones ruidosas de f(x) (analogo a los 'anchor points' de alta fidelidad del paper)
x_f = np.sort(np.random.uniform(0, 1, n_f)).reshape(-1, 1)
noise_f = 0.05
y_f = exact_f(x_f) + noise_f * np.random.randn(*x_f.shape)

# Punto ancla en u(x) (informacion adicional sobre la propia solucion, como en el paper)
x0 = np.array([[0.0]])
y0 = exact_u(x0) + 0.01 * np.random.randn(*x0.shape)

## 2. Kernel fisicamente informado (Eq. 3-4): $g$ = kernel exponencial cuadratico sobre $u$; $k=\mathcal{L}_x\mathcal{L}_{x'}g$ sobre $f$

Con $\mathcal{L}_x=\partial_x$ (derivada), y $g(x,x';\theta)=\sigma^2\exp(-\tfrac{w}{2}(x-x')^2)$, las derivadas del kernel se obtienen analiticamente (formulas estandar del kernel SE derivado).

In [ ]:
def g_kernel(x, xp, sigma2, w):
    d = x - xp.T
    return sigma2 * np.exp(-0.5 * w * d**2)

def dg_dxp(x, xp, sigma2, w):
    """d/dx' g(x,x') -- cov(u(x), f(x')) ya que f=Lx[u]=du/dx evaluado en x' (Eq. 9-10 adaptada)."""
    d = x - xp.T
    return w * d * g_kernel(x, xp, sigma2, w)

def d2g_dxdxp(x, xp, sigma2, w):
    """k(x,x') = d/dx d/dx' g(x,x') -- cov(f(x), f(x')) (Eq. 4)."""
    d = x - xp.T
    return sigma2 * w * (1 - w * d**2) * np.exp(-0.5 * w * d**2)

## 3. Entrenamiento (Eq. 5-6): maximizar la verosimilitud marginal para $(\sigma^2, w, \sigma_{n_0}^2, \sigma_{n_f}^2)$

In [ ]:
def build_K(sigma2, w, sn0_2, snf_2):
    K00 = g_kernel(x0, x0, sigma2, w) + sn0_2 * np.eye(len(x0))
    K0f = dg_dxp(x0, x_f, sigma2, w)
    Kff = d2g_dxdxp(x_f, x_f, sigma2, w) + snf_2 * np.eye(len(x_f))
    K = np.block([[K00, K0f], [K0f.T, Kff]])
    return K

y = np.vstack([y0, y_f])

def neg_log_marg_lik(log_params):
    sigma2, w, sn0_2, snf_2 = np.exp(log_params)
    K = build_K(sigma2, w, sn0_2, snf_2) + 1e-8 * np.eye(len(y))
    try:
        L = np.linalg.cholesky(K)
    except np.linalg.LinAlgError:
        return 1e10
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    nlml = 0.5 * y.T @ alpha + np.sum(np.log(np.diag(L))) + 0.5 * len(y) * np.log(2 * np.pi)
    return nlml.item()

x0_init = np.log([1.0, 5.0, 1e-4, noise_f**2])
res = minimize(neg_log_marg_lik, x0_init, method='Nelder-Mead',
                options={'maxiter': 2000, 'xatol': 1e-6, 'fatol': 1e-6})
sigma2_opt, w_opt, sn0_2_opt, snf_2_opt = np.exp(res.x)
print(f'Hiperparametros optimizados: sigma2={sigma2_opt:.4f}, w={w_opt:.4f}, '
      f'sn0_2={sn0_2_opt:.2e}, snf_2={snf_2_opt:.4f}')

## 4. Prediccion (Eq. 15): distribucion posterior de $u(x)$ dado solo las observaciones ruidosas de $f$

In [ ]:
K = build_K(sigma2_opt, w_opt, sn0_2_opt, snf_2_opt) + 1e-8 * np.eye(len(y))
K_inv = np.linalg.inv(K)

x_test = np.linspace(0, 1, 200).reshape(-1, 1)

a_u0 = g_kernel(x_test, x0, sigma2_opt, w_opt)          # cov(u(x*), u(x0))
a_uf = dg_dxp(x_test, x_f, sigma2_opt, w_opt)             # cov(u(x*), f(x_f))
a = np.hstack([a_u0, a_uf])

u_mean = a @ K_inv @ y
u_var = g_kernel(x_test, x_test, sigma2_opt, w_opt).diagonal().reshape(-1, 1) - \
        np.sum((a @ K_inv) * a, axis=1, keepdims=True)
u_std = np.sqrt(np.clip(u_var, 0, None))

## 5. Resultados (cf. Fig. 1(C) del paper): media posterior +/- 2 desviaciones estandar

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(x_test, exact_u(x_test), 'k-', label='Solucion exacta u(x)', linewidth=1.5)
plt.plot(x_test, u_mean, 'r--', label='Media posterior del GP')
plt.fill_between(x_test.flatten(), (u_mean - 2 * u_std).flatten(), (u_mean + 2 * u_std).flatten(),
                  color='orange', alpha=0.3, label='+/- 2 desviaciones estandar')
plt.scatter(x0, y0, color='red', marker='s', zorder=5, label='Punto ancla en u')
plt.xlabel('x'); plt.ylabel('u(x)')
plt.title("Inferencia de u(x) a partir de observaciones ruidosas de f(x)=u'(x) via GP")
plt.legend()
plt.show()

err = 100 * np.linalg.norm(u_mean - exact_u(x_test)) / np.linalg.norm(exact_u(x_test))
print(f'Error relativo L2 de la media posterior: {err:.2f}%')